# HTTP Utilities (`http_utilities`)

El módulo `http_utilities` (`ds_utils.io.http_utilities`) proporciona funciones para la creación, configuración y gestión de sesiones HTTP homogéneas y seguras mediante la librería `requests`.

### Características principales:
- **`get_http_session`**: Crea y configura una instancia de `requests.Session` lista para interactuar con servicios web y APIs REST.
- **Cabeceras estándar preconfiguradas**: Establece automáticamente la cabecera `'accept': 'application/json'` para un intercambio estructurado de datos.
- **Autenticación Bearer segura mediante variables de entorno**: Permite inyectar de forma transparente tokens de autorización `Bearer <token>` cargados directamente desde el entorno (`token_env_var`), evitando exponer credenciales en el código fuente o en cuadernos de trabajo.
- **Validación estricta de credenciales**: Comprueba que las variables de entorno existan y no estén vacías, alertando tempranamente de fallos de configuración mediante excepciones claras.
- **Reutilización de conexiones (Connection Pooling)**: Al utilizar `requests.Session`, las conexiones TCP subyacentes se mantienen abiertas y se reutilizan entre peticiones al mismo host, optimizando el rendimiento y la latencia.

### Contenido del Cuaderno:
1. **Configuración e Importaciones**: Preparación del entorno de ejecución.
2. **Creación de una Sesión HTTP Básica**: Sesión estándar sin autenticación con cabeceras por defecto.
3. **Envío de Peticiones con la Sesión**: Uso de la sesión configurada para consultar endpoints de datos.
4. **Sesión HTTP con Autenticación Bearer**: Inyección automática de tokens de autorización desde variables de entorno.
5. **Validación y Manejo de Errores**: Comportamiento ante variables de entorno ausentes o vacías.
6. **Personalización Avanzada de la Sesión**: Inyección de cabeceras personalizadas y persistencia entre peticiones.
7. **Cierre de Sesión y Liberación de Recursos**: Buenas prácticas de gestión del ciclo de vida de la sesión.

## 1. Configuración e Importaciones

Habilitamos la recarga automática de módulos e importamos las librerías necesarias.

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import os
import requests
from requests.adapters import BaseAdapter
from requests.models import Response

from ds_utils.io.http_utilities import get_http_session

print(f"Versión de requests: {requests.__version__}")

Versión de requests: 2.34.2


## 2. Creación de una Sesión HTTP Básica (sin autenticación)

Al llamar a `get_http_session()` sin argumentos, se inicializa una sesión de `requests.Session` configurada con la cabecera `'accept': 'application/json'`.

In [2]:
# Creación de una sesión básica sin autenticación
session = get_http_session()

print(f"Tipo de objeto devuelto: {type(session)}")
print(f"Cabeceras configuradas en la sesión:\n{dict(session.headers)}")

Tipo de objeto devuelto: <class 'requests.sessions.Session'>
Cabeceras configuradas en la sesión:
{'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate', 'accept': 'application/json', 'Connection': 'keep-alive'}


## 3. Envío de Peticiones con la Sesión

Utilizamos la sesión para realizar peticiones HTTP. Las cabeceras configuradas a nivel de sesión se aplican automáticamente a todas las llamadas.

> **Nota:** Para garantizar la reproducibilidad y ejecución sin depender de conexión a internet o servicios externos que puedan variar, utilizamos un `MockAPIAdapter` de `requests` que responde localmente con datos simulados.

In [3]:
# Adaptador local para simular respuestas de API de forma reproducible y sin red externa
class MockAPIAdapter(BaseAdapter):
    def send(self, request, **kwargs):
        res = Response()
        res.status_code = 200
        res.request = request
        res.headers["Content-Type"] = "application/json"
        if "telemetry" in request.url:
            res._content = b'{"status": "ok", "readings": [{"sensor": "temp", "val": 22.4}]}'
        else:
            res._content = b'{"service": "data-api", "version": "1.0.0", "status": "online"}'
        return res

    def close(self):
        pass

# Montamos el adaptador simulado en el prefijo de la API
session.mount("https://api.internal", MockAPIAdapter())

# Realizamos una petición GET
response = session.get("https://api.internal/telemetry")

print(f"Código de estado HTTP: {response.status_code}")
print(f"Respuesta JSON recibida: {response.json()}")
print(f"Cabecera 'accept' transmitida en la petición: {response.request.headers.get('accept')}")

Código de estado HTTP: 200
Respuesta JSON recibida: {'status': 'ok', 'readings': [{'sensor': 'temp', 'val': 22.4}]}
Cabecera 'accept' transmitida en la petición: application/json


## 4. Sesión HTTP con Autenticación Bearer Token

Para interactuar con APIs protegidas (p. ej. OAuth2, JWT o claves de servicio), `get_http_session` acepta el parámetro `token_env_var`.
Este parámetro especifica el nombre de la variable de entorno que contiene el token, inyectando automáticamente la cabecera:

`Authorization: Bearer <token>`

In [4]:
# Configuramos una variable de entorno de prueba para el token
ENV_TOKEN_NAME = "DATA_SERVICE_BEARER_TOKEN"
os.environ[ENV_TOKEN_NAME] = "sec_tok_9876543210_abcdef"

# Creamos la sesión autenticada pasando el nombre de la variable
auth_session = get_http_session(token_env_var=ENV_TOKEN_NAME)

print(f"Cabecera Authorization configurada: {auth_session.headers.get('Authorization')}")
print(f"Cabecera accept configurada:       {auth_session.headers.get('accept')}")

Cabecera Authorization configurada: Bearer sec_tok_9876543210_abcdef
Cabecera accept configurada:       application/json


### Verificación de la transmisión del token en peticiones autenticadas

Comprobamos que las peticiones emitidas por `auth_session` transmiten de forma transparente la cabecera `Authorization`.

In [5]:
# Montamos el adaptador de prueba en la sesión autenticada
auth_session.mount("https://api.internal", MockAPIAdapter())

# Enviamos la petición
auth_response = auth_session.get("https://api.internal/secure-data")

print(f"Respuesta recibida: {auth_response.json()}")
print(f"Cabecera Authorization enviada en el request: {auth_response.request.headers.get('Authorization')}")

# Limpiamos la variable de entorno utilizada para la prueba
del os.environ[ENV_TOKEN_NAME]
print(f"Variable '{ENV_TOKEN_NAME}' eliminada del entorno: {ENV_TOKEN_NAME not in os.environ}")

Respuesta recibida: {'service': 'data-api', 'version': '1.0.0', 'status': 'online'}
Cabecera Authorization enviada en el request: Bearer sec_tok_9876543210_abcdef
Variable 'DATA_SERVICE_BEARER_TOKEN' eliminada del entorno: True


## 5. Validación y Manejo de Errores en Variables de Entorno

`get_http_session` valida de forma estricta que la variable de entorno solicitada exista y contenga un valor no vacío.
Si la variable no está definida o contiene una cadena vacía, se lanza una excepción `ValueError`, alertando inmediatamente sobre problemas de configuración en el entorno de ejecución.

In [6]:
# Caso 1: Variable de entorno no definida
try:
    get_http_session(token_env_var="NON_EXISTENT_SECRET_TOKEN")
except ValueError as exc:
    print(f"Excepción capturada (variable no definida):\n-> {exc}")

print("-" * 60)

# Caso 2: Variable de entorno definida pero con valor vacío
os.environ["EMPTY_SECRET_TOKEN"] = ""
try:
    get_http_session(token_env_var="EMPTY_SECRET_TOKEN")
except ValueError as exc:
    print(f"Excepción capturada (variable vacía):\n-> {exc}")
finally:
    del os.environ["EMPTY_SECRET_TOKEN"]

Excepción capturada (variable no definida):
-> Environment variable 'NON_EXISTENT_SECRET_TOKEN' is not defined or contains an empty value.
------------------------------------------------------------
Excepción capturada (variable vacía):
-> Environment variable 'EMPTY_SECRET_TOKEN' is not defined or contains an empty value.


## 6. Personalización Avanzada de la Sesión

La sesión devuelta es un objeto estándar `requests.Session`. Es posible ampliarla con configuraciones específicas como cabeceras personalizadas de cliente, parámetros de timeout o reintentos.

In [7]:
custom_session = get_http_session()

# Añadimos cabeceras personalizadas adicionales a nivel de sesión
custom_session.headers.update({
    "User-Agent": "ds-utils-data-client/1.0",
    "X-Client-ID": "data-pipeline-01",
    "X-Request-Timeout": "30",
})

print("Cabeceras completas configuradas en la sesión personalizada:")
for key, value in custom_session.headers.items():
    print(f"  {key}: {value}")

Cabeceras completas configuradas en la sesión personalizada:
  User-Agent: ds-utils-data-client/1.0
  Accept-Encoding: gzip, deflate
  accept: application/json
  Connection: keep-alive
  X-Client-ID: data-pipeline-01
  X-Request-Timeout: 30


## 7. Cierre de Sesión y Liberación de Recursos

Es una buena práctica invocar `session.close()` al finalizar el uso del cliente HTTP para cerrar las conexiones TCP abiertas y liberar sockets del sistema operativo.

In [8]:
# Cerramos las sesiones HTTP abiertas durante el notebook
session.close()
auth_session.close()
custom_session.close()

print("Todas las sesiones HTTP se cerraron correctamente y los recursos del sistema fueron liberados.")

Todas las sesiones HTTP se cerraron correctamente y los recursos del sistema fueron liberados.
